In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale (0,1) => normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root='./data',train=True,download=True,transform=transform)
testset = CIFAR10(root='./data',train=False,download=True,transform=transform)



100%|██████████| 170M/170M [22:19<00:00, 127kB/s]


In [5]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(trainset,batch_size=64)


In [6]:
### Build the cnn

In [11]:
class CNN(nn.Module):
  def __init__(self):
      super(CNN,self).__init__()

      self.conv_layers = nn.Sequential(
          nn.Conv2d(3,32,kernel_size=3,padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2,2), # kernal size = 2, stride = 2

          nn.Conv2d(32,64,kernel_size=3,padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2,2), # kernal size = 2, stride = 2

          nn.Conv2d(64,128,kernel_size=3,padding=1),
          nn.ReLU(),
          nn.MaxPool2d(2,2), # kernal size = 2, stride = 2
      )

      self.fc_layers = nn.Sequential(
          nn.Linear(4*4*128,256),
          nn.ReLU(),

          nn.Linear(256,10)
      )

  def forward(self,x):
      x = self.conv_layers(x)
      x = x.view(x.size(0),-1) # flattening
      x = self.fc_layers(x)

      return x

In [12]:
model = CNN()

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [14]:
## Training the CNN

In [15]:
epochs = 10

for epoch in range(epochs):
  epoch_training_loss = 0.0

  for images,labels in trainloader:
    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs,labels) # loss fnx
    loss.backward() # BP
    optimizer.step()

    epoch_training_loss += loss.item()
    model.forward(images) # FP

  print(f"Epoch {epoch+1}/{epochs}, Training Loss: {epoch_training_loss/len(trainloader)}")

Epoch 1/10, Training Loss: 1.3900643898093181
Epoch 2/10, Training Loss: 0.9565920857212428
Epoch 3/10, Training Loss: 0.7694564877873491
Epoch 4/10, Training Loss: 0.6474368643882634
Epoch 5/10, Training Loss: 0.548821882709213
Epoch 6/10, Training Loss: 0.4566442493892387
Epoch 7/10, Training Loss: 0.37588347402184513
Epoch 8/10, Training Loss: 0.30448472811399824
Epoch 9/10, Training Loss: 0.2393427415062552
Epoch 10/10, Training Loss: 0.19525123502859068


In [16]:
# Evaluate our cnn

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
  for images,labels in testloader:
    outputs = model(images)
    _,predicted = torch.max(outputs,1)

    total_labels += labels.size(0)
    correct_labels += (predicted == labels).sum().item()

print(f"epoch={epoch+1}/{epoch} & loss={epoch_training_loss/len(trainloader)}")

epoch=10/9 & loss=0.19525123502859068
